# 실전 데이터 분석 62: 주간 자기주도 공부 시간, 수면 시간 및 부모 지원 수준 대비 최종 성적 향상 상관성 분석

> **📥 [실습 주피터 노트북(.ipynb) 다운로드](practice.ipynb)**

## 📌 강의 개요 (30분 완성)

![코믹 일러스트](img/intro_comic.png)

학생들의 자기주도 학습 패턴 및 부모의 학업 관심도를 모은 학업 성취도 데이터셋입니다. 학생들의 공부 시간 분포를 파악하고, 학습 투입량과 성적(FinalGrade) 간의 인과적 트렌드를 부모 지원(ParentalSupport) 환경별로 다각도 분산 분석합니다.

**학습 목표:**
* **결측치 평균 대치 (`fillna`):** 출석률 컬럼에 일부 발생한 유실값을 학생들의 전체 평균 출석률로 보완합니다.
* **밀도 오버레이 분포 차트 (`histplot`):** 주간 스스로 공부하는 시간의 편차를 빈(Bin) 히스토그램과 KDE 분포 곡선으로 부드럽게 요약합니다.

---

## 🧐 실무 도메인 지식 가이드 (Domain Knowledge Guide)

> **🎓 에듀테크 및 교육 (EdTech & Education)**
> 교육 데이터 분석은 온라인 학습 로그, 학생 성적 성취도, 중도 탈락 위험(Dropout)을 식별해 개인화된 성취 성장을 지원하는 분야입니다.
>
> * **학업 중도 탈락(Dropout) 예측**: 학생의 마지막 접속 주기, 인터랙션 클릭 감소 성향 등을 조기 진단하여 보충 처방 학습을 유도합니다.
> * **학습 몰입 지표(Engagement)**: 뷰어 응답 반응성, 동영상 스키핑 분석으로 주의 분산 한계 지점을 식별해 맞춤 교과 콘텐츠를 기획합니다.
> * **장학 지원 성과 측정**: 지원금 예산 배분 대비 학점 상승 성취 효과(ROI)의 통계적 정성 분석을 통해 예산 효율성을 제고합니다.

---

## Step 1: 데이터 구조 살펴보기 (Data Overview)

![Step 1 데이터 구조 개념도](img/step1_dataset.svg)

`csv_data` 폴더에 준비해 둔 `student_performance.csv` 파일을 판다스로 불러옵니다.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 그래프 설정 (한글 폰트 및 마이너스 기호 깨짐 방지)
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

# 로컬 CSV 파일 불러오기
df = pd.read_csv('./student_performance.csv')

# 데이터 구조 및 첫 5행 확인
df = pd.read_csv('./student_performance.csv')
print(df.info())
print(df.head())


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   StudentID        1000 non-null   int64  
 1   StudyHours       1000 non-null   float64
 2   AttendanceRate   988 non-null    float64
 3   SleepHours       1000 non-null   float64
 4   ParentalSupport  1000 non-null   str    
 5   FinalGrade       1000 non-null   float64
dtypes: float64(4), int64(1), str(1)
memory usage: 47.0 KB
StudentID  StudyHours  AttendanceRate  SleepHours ParentalSupport  FinalGrade
0     620001         3.1            89.2         8.1          Medium        62.7
1     620002        18.1            90.0         7.4            High        82.5
2     620003        29.9            82.9         7.1            High        94.6
3     620004        15.6            85.0         8.3            High        81.9
4     620005        22.8            86.7         7.3          Medium        86.2
> ```

### 💡 코드 딥다이브 (Code Deep Dive)
**주요 분석 대상 컬럼:**
* `StudentID`: 학생 고유 식별 번호
* `StudyHours`: 주간 자기주도 학습 시간 (시간 단위)
* `AttendanceRate`: 주간 수업 출석률 (%) (결측치 존재)
* `SleepHours`: 일평균 수면 시간 (시간 단위)
* `ParentalSupport`: 부모의 학습 관심도 및 지원 등급 (High, Medium, Low)
* `FinalGrade`: 학기말 최종 교과 성적 (100점 만점 기준)

---

## Step 2: 전처리와 결측치 정제 (Preprocess)

![Step 2 데이터 정제 개념도](img/step2_missing_values.svg)

현실의 데이터는 항상 누락이 있거나 유효성 정제가 필요합니다. 데이터 전처리 단계에서 결측 상태를 확인하고 올바르게 보정합니다.


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 정제 전 결측치 확인 ---")
print(df.isnull().sum())

# 2. 결측치 전처리 및 대치 수행
mean_att = df['AttendanceRate'].mean()
df['AttendanceRate'] = df['AttendanceRate'].fillna(mean_att)
print(df.isnull().sum())


--- 정제 전 결측치 확인 ---
StudentID           0
StudyHours          0
AttendanceRate     12
SleepHours          0
ParentalSupport     0
FinalGrade          0

--- 정제 후 결측치 확인 ---
StudentID          0
StudyHours         0
AttendanceRate     0
SleepHours         0
ParentalSupport    0
FinalGrade         0
> ```

### 💡 분석가의 통찰 (Analyst's Insight)
* **평균 대치를 선택하는 실무적 근거:** 학기 중 시스템 연동 장애 등으로 유실된 출석률 데이터를 보완합니다. 전체 학생의 출석률 평균은 약 85% 대역에서 안정적인 등급을 보이고 있으므로, 이 평균치를 결측값에 대입하여 연속성을 유지하고 다변수 예측 분석의 안정성을 높입니다.

---

## Step 3: 단변수 분포 분석 (Univariate EDA)

![Step 3 시각화 개념도](img/step3_visualization.svg)

가장 먼저 핵심 변수가 전체 데이터에서 어떤 빈도와 분포를 가졌는지 단일 변수 시각화를 통해 파악해 봅니다.


In [ ]:
plt.figure(figsize=(8, 5))

# 단변수 분석 그래프 생성
sns.histplot(data=df, x='StudyHours', bins=15, kde=True, color='blue')
plt.title('주간 학습 시간(Study Hours) 분포', fontsize=14, fontweight='bold')
plt.show()


### 💡 시각화 차트 읽는 법 & 인사이트
* **고른 학업 투입 시간의 정규분포 관찰:** 주간 공부 시간 분포 히스토그램을 보면 주 15시간 내외를 정점으로 양옆으로 완만히 떨어지는 정규분포 양상을 띱니다. 아주 열심히 하는 30시간 이상의 고학습 집단과 5시간 미만의 저학습 집단이 양단에 대칭 분포하고 있습니다.

---

## Step 4: 다변수 상관관계 및 이상치 분석 (Multivariate EDA)

![Step 4 상관관계 분석 개념도](img/step4_multivariate.svg)

두 개 이상의 변수를 동시에 결합하여, 조건에 따른 수치 차이나 독립 변수와 종속 변수 간의 통계적 경향을 분석합니다.


In [ ]:
plt.figure(figsize=(9, 6))

# 다변수 분석 그래프 생성
sns.scatterplot(data=df, x='StudyHours', y='FinalGrade', hue='ParentalSupport', palette='Set1', alpha=0.8)
plt.title('학습 시간 대비 성적 분포와 부모 지원 상관성', fontsize=14, fontweight='bold')
plt.show()


### 💡 코드 딥다이브 & 비즈니스 통찰 (Analyst's Insight)
* **부모 관심도가 주는 성적 가속 시너지 규명:** 공부 시간(X축)과 최종 성적(Y축)이 뚜렷한 우상향 선형 상관성을 나타냅니다. 특히 흥미로운 점은 동일한 공부 시간을 투입하더라도 부모 지원 등급이 'High'(빨간 점 계열)인 학생 그룹이 'Low'(파란 점 계열) 그룹보다 성적 대역의 상단부를 선점하고 있어, 가정의 정서적 지원 환경이 중요한 변수임을 공간적으로 확인시켜 줍니다.

---

## Step 5: 통계적 직관과 해석 (Statistical Logic)

> 💡 **[피어슨 상관계수(Pearson Correlation Coefficient)의 수학적 이해]**
> 공부 시간과 최종 성적 간의 선형 관계 강도를 계량화하는 가장 표준적인 지표는 **피어슨 상관계수($r$)**입니다. 두 변수의 공분산을 각각의 표준편차 곱으로 나누어 구합니다.
> $$ r = \frac{\sum (X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum (X_i - \bar{X})^2 \sum (Y_i - \bar{Y})^2}} $$
> 
> * 피어슨 상관계수 $r$은 **$-1$부터 $+1$ 사이**의 값을 취합니다. 
> * 본 실습의 시각화 결과처럼 $r \approx 0.82$ 수준의 강한 양의 상관성을 얻는다면, 이는 공부 시간 증가와 성적 향상이 매우 정직하고 조밀하게 궤도를 같이하고 있음을 의미합니다.
> * 다만, 상관성은 두 요인의 물리적인 인과관계를 100% 보장하는 것이 아니므로, 수면 시간과 같은 교란 요인의 기여도를 다중 회귀 모델로 통제한 뒤 독립적인 관계를 도출하는 EDA 해석 습관이 요구됩니다.

---

## 🎯 30분 강의 마무리 및 심화 과제

오늘 우리는 실전 데이터셋을 분석하여 판다스로 데이터를 가공 및 정제하고, 시각화를 활용하여 핵심 변수 간의 통계적 유의성을 검증했습니다. 데이터 속에서 숨겨진 패턴을 올바른 시각으로 탐색하는 능력이 데이터 사이언티스트의 가장 강력한 무기입니다.

### 📝 심화 과제 (Advanced Challenge)
1. **수면 시간(SleepHours)과 성적(FinalGrade)의 상관계수 계산:** 잠을 너무 적게 자면서 공부하는 역효과가 있는지 두 요인 간의 피어슨 상관계수를 판다스 `.corr()`로 구하고 직관을 적어 보세요.
2. **부모 지원 유형별 평균 성적 집계:** `ParentalSupport`별 평균 `FinalGrade`와 공부시간을 요약 테이블로 도출하고 어떤 그룹의 학업 효율이 극대화되었는지 분석하세요.
